# Create Vector DB

In [1]:
import glob
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Load all documents in knowledge-base

folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = os.path.basename(folder)
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 76 documents


In [2]:
# inspect loaded documents
documents[0]

Document(metadata={'source': 'knowledge-base\\company\\about.md', 'doc_type': 'company'}, page_content="# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.\n\nHowever, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have built a

In [3]:
# Divide documents into chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter tries to split in below order:
# - paragraph i.e. \n\n
# - new lines i.e. \n
# - words i.e. " "
# - characters i.e. ""
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")

Divided into 413 chunks


In [8]:
# inspect chunks

chunks[0]

Document(metadata={'source': 'knowledge-base\\company\\about.md', 'doc_type': 'company'}, page_content='# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.')

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Convert chunks into vectors and add to vectorDB

db_name = "vector_db"

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

if os.path.exists(db_name): # if vector DB already exists, delete it
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)

# Note, each chunk is converted to a vector. Hence, n_chunks == n_vectors
print(f"Vector DB created with name {db_name} and {vectorstore._collection.count()} number of vectors")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector DB created with name vector_db and 413 number of vectors


In [ ]:
# Visualize vectors

# limit=1 returns only 1 record
sample_embedding = vectorstore._collection.get(limit=1, include=["embeddings"])["embeddings"][0]
print(f"Each vector is of dims {len(sample_embedding)}")

Each vector is of dims 384


In [17]:

result = vectorstore._collection.get(include=["embeddings", "documents", "metadatas"])
type(result["embeddings"])

numpy.ndarray

In [20]:
# Visualize data

from sklearn.manifold import TSNE
import plotly.graph_objects as go
import numpy as np

# prepare
result = vectorstore._collection.get(include=["embeddings", "documents", "metadatas"])
vectors = result["embeddings"]
documents = result["documents"]
metadatas = result["metadatas"]
doc_types = [metadata["doc_type"] for metadata in metadatas]
colors = [["blue", "green", "red", "orange"] \
            [["products", "employees", "contracts", "company"].index(doc_type)] \
            for doc_type in doc_types]

# t-distributed stochastic neighbour embedding
# random state is needed because TSNE starts with data distributed at random location before organizing.
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Plot the graph
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y= reduced_vectors[:, 1],
    mode="markers", # to plot dots only
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)], # plotly treats <br> as newline
    hoverinfo="text"
)])

# polish non data parts
fig.update_layout(title="2D Chroma vector store visualization",
    scene=dict(xaxis_title="x", yaxis_title="y"),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

# Use vectorDB to provide additional context to LLM

In [ ]:
# Get vectorDB instance as retriever

embeddings = HuggingFaceEmbeddings(model_name="all-miniLM-L6-v2") # Should be same encoder model used to create vectorDB
vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings) # db_name is already initialized to vector_db
retriever = vectorstore.as_retriever()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# verify retriever
retriever.invoke("Who is Avery") # returns related chunks

[Document(id='c403c59d-2eee-4971-8f98-daeb27ec8e47', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [26]:
# Get an LLM

from langchain_ollama import ChatOllama

llm = ChatOllama(temperature=0, model="gpt-oss:120b-cloud")

In [27]:
# Verify llm
llm.invoke("Who is Avery ?")

AIMessage(content='**Avery** can refer to many different people, characters, or even just a first‐name or surname. Below are a few of the most common contexts where the name “Avery” shows up. If you had a specific Avery in mind, let me know and I can dive deeper!\n\n---\n\n## 1.\u202fAvery as a Given Name  \n- **Origin & Meaning** –\u202fDerived from the Old English *Æþelfrith* (“noble” + “peace”) or the Old French *Averi* (a variant of Alfred). It’s traditionally masculine but has become widely used as a gender‑neutral name in the United States and other English‑speaking countries.  \n- **Popularity** –\u202fIn the U.S., “Avery” has been a top‑100 name for both boys and girls since the early 2000s (e.g., #27 for boys and #33 for girls in 2022).\n\n## 2.\u202fFamous People Named Avery  \n\n| Field | Notable “Avery” | Quick Bio |\n|-------|-----------------|-----------|\n| **Sports** | **Avery\u202fBradley** (b.\u202f1993) – NBA shooting guard, 2023 NBA\u202fChampion with the Denver Nug

In [28]:
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

# Get context from vectorDB and pass it to LLM

SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

def answer_question(question: str, history):
    # Retrieve relevant context from vectorDB and append to system prompt
    docs = retriever.invoke(question) # Get relevant docs from vectorDB
    context = "\n\n".join([doc.page_content for doc in docs]) # Join all relevant docs
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context) # Update system prompt with relevant context

    # Provide content for role=system and role=user
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])

    return response.content


gr.ChatInterface(answer_question).launch()

c:\ed\projects\rag_langchain\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# Advance

In [ ]:
from langchain_core.messages import convert_to_messages

# Add to context earlier user history as well

def answer_question(question: str, history):
    print(history)
    # Prefix question with old user history before retrieving context from vectorDB 
    history_question = "\n".join([m["content"] for m in history if m["role"] == "user"]) + "\n" + question
    docs = retriever.invoke(history_question) # Note, getting relevant context for history + question
    context = "\n\n".join([doc.page_content for doc in docs]) # Join all relevant docs
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context) # Update system prompt with relevant context

    # Invoke LLM
    messages = [SystemMessage(content=system_prompt)] # system prompt with additional context
    messages.extend(convert_to_messages(history)) # history of user and assistant
    messages.append(HumanMessage(content=question)) # user question
    response = llm.invoke(messages)

    return response.content

gr.ChatInterface(fn=answer_question, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


[]
[{'role': 'user', 'metadata': None, 'content': 'Who is Avery ?', 'options': None}, {'role': 'assistant', 'metadata': None, 'content': 'Avery Lancaster is the Co‑Founder and Chief Executive Officer (CEO) of Insurellm.  \n\n- **Background**: Born on March\u202f15\u202f1985, Avery co‑founded Insurellm in 2015 after serving as a Senior Product Manager at Innovate Insurance Solutions (2013‑2015), where she helped build cutting‑edge insurance products for the tech sector.  \n\n- **Role at Insurellm**: As CEO, she drives the company’s overall vision, strategy, and growth. Her leadership is recognized for innovation in insurance‑tech, strong risk‑management practices, and fostering a culture of diversity, inclusion, and work‑life balance.  \n\n- **Key Achievements**  \n  * Guided Insurellm to become a market leader with personalized insurance solutions.  \n  * Launched two successful products in 2018 that boosted market share.  \n  * Championed community outreach and financial‑literacy prog